# Task 1: environment and repository check

Run from the repository root after cloning and installing (`requirements.txt`, then `pip install -e . --no-deps`; do not reinstall torch on Kaggle). The committed copy was **executed on the training server** (2x RTX 6000 Ada, shared) that replaced Kaggle T4x2 for all runs. The same cells run unchanged on a Kaggle T4x2 notebook.

In [1]:
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')  # run everything from the repository root
print(os.getcwd())

/home/sk10741/genaiPA2/pa2


In [2]:
!nvidia-smi

Sat Oct 10 16:39:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.207                Driver Version: 570.207        CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX 6000 Ada Gene...    On  |   00000000:41:00.0 Off |                  Off |
| 60%   84C    P0            204W /  300W |    3551MiB /  49140MiB |     63%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

|    0   N/A  N/A            3715      G   /usr/lib/xorg/Xorg                        4MiB |
|    0   N/A  N/A          746279      C   python                                 3540MiB |
|    1   N/A  N/A            3715      G   /usr/lib/xorg/Xorg                        4MiB |
+-----------------------------------------------------------------------------------------+


In [3]:
import platform, torch, transformers, datasets, trl, peft
print('python      ', platform.python_version())
print('torch       ', torch.__version__, '| CUDA', torch.version.cuda)
print('transformers', transformers.__version__)
print('datasets    ', datasets.__version__)
print('trl         ', trl.__version__, '| peft', peft.__version__)
n = torch.cuda.device_count()
print('GPUs visible:', n)
for i in range(n):
    p = torch.cuda.get_device_properties(i)
    print(f'  cuda:{i}', p.name, f'{p.total_memory / 2**30:.1f} GiB')
assert n >= 2, 'expected two GPUs (policy and reference)'

/home/sk10741/genaiPA2/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


python       3.10.12
torch        2.6.0+cu124 | CUDA 12.4
transformers 4.57.6
datasets     4.8.5
trl          1.14.1 | peft 0.21.2
GPUs visible: 2
  cuda:0 NVIDIA RTX 6000 Ada Generation 47.4 GiB
  cuda:1 NVIDIA RTX 6000 Ada Generation 47.4 GiB


In [4]:
# Public adapter smoke tests + extra edge-case tests (CPU only, no downloads)
import sys
!PYTHONPATH=. {sys.executable} -m pytest -q tests/

..

.........

......

....                                                    [100%]
21 passed in 2.41s


In [5]:
# Fixed evaluation subsets used by every comparison
import json
for f in ['eval_sets/hh_rlhf_eval_300_seed42.json', 'eval_sets/alpaca_eval_300_seed42.json', 'eval_sets/persona_eval_40.json']:
    rows = json.load(open(f)); print(f, len(rows), rows[0]['id'])

eval_sets/hh_rlhf_eval_300_seed42.json 300 hh-eval-42-0000
eval_sets/alpaca_eval_300_seed42.json 300 alpaca-eval-42-0000
eval_sets/persona_eval_40.json 40 persona-eval-cooking-00
